<a href="https://colab.research.google.com/github/kasettisiva/R_training_summer_workshop_2026/blob/main/R_Practice_Solutions_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# R Practice — SOLUTIONS — Summer 2026
## Data Analysis with Real-World Flight Data

> **Instructor copy** — Please do not distribute to attendees before the session.

---

## ▶ Step 0 — Setup

In [ ]:
install.packages(c("ggplot2", "dplyr", "tidyr", "Metrics"))
library(ggplot2); library(dplyr); library(tidyr); library(Metrics)
cat("✅ Packages ready.\n")

---
## Step 1 · Load & Inspect the Data

In [ ]:
url <- "https://raw.githubusercontent.com/tidyverse/nycflights13/master/data-raw/flights.csv"
flights <- read.csv(url, stringsAsFactors = FALSE)
cat("Rows:", nrow(flights), "| Columns:", ncol(flights), "\n")

**Solution 1.1**

In [ ]:
class(flights)       # "data.frame"
nrow(flights)        # 336,776
ncol(flights)        # 19
dim(flights)         # 336776 x 19

**Solution 1.2**

In [ ]:
head(flights)
str(flights)

**Solution 1.3**

In [ ]:
summary(flights)
# dep_time, dep_delay, arr_time, arr_delay, air_time all have NAs (cancelled flights)

---
## Step 2 · Handle Missing Values

**Solution 2.1**

In [ ]:
sum(is.na(flights$dep_delay))   # 8,255 cancelled flights

**Solution 2.2**

In [ ]:
flights_clean <- flights %>%
  filter(!is.na(dep_delay), !is.na(arr_delay))
nrow(flights_clean)   # 327,346 rows remain

**Solution 2.3**

In [ ]:
cancelled <- sum(is.na(flights$dep_delay))
pct <- round(cancelled / nrow(flights) * 100, 2)
cat("Cancelled flights:", pct, "%\n")   # ~2.45%

---
## Step 3 · Filter & Subset

**Solution 3.1**

In [ ]:
ua_flights <- flights_clean %>% filter(carrier == "UA")
nrow(ua_flights)   # 58,036 United Airlines flights

**Solution 3.2**

In [ ]:
very_late <- flights_clean %>% filter(dep_delay > 120)
nrow(very_late)

# Which carrier had the most?
very_late %>%
  count(carrier, sort = TRUE) %>%
  head(5)

**Solution 3.3**

In [ ]:
flights_sub <- flights_clean %>%
  select(carrier, origin, dest, dep_delay, arr_delay, distance)

**Solution 3.4**

In [ ]:
flights_sub <- flights_sub %>%
  mutate(delay_diff = arr_delay - dep_delay)
# Negative delay_diff means the flight made up time in the air
head(flights_sub)

---
## Step 4 · Group & Summarize

**Solution 4.1**

In [ ]:
flights_clean %>%
  group_by(carrier) %>%
  summarize(
    mean_delay   = round(mean(dep_delay), 1),
    median_delay = round(median(dep_delay), 1),
    n_flights    = n()
  ) %>%
  arrange(desc(mean_delay))

**Solution 4.2**

In [ ]:
flights_clean %>%
  group_by(origin) %>%
  summarize(
    n_flights     = n(),
    mean_delay    = round(mean(dep_delay), 1),
    pct_over_15   = round(mean(dep_delay > 15) * 100, 1)
  )

**Solution 4.3**

In [ ]:
flights_clean %>%
  group_by(month) %>%
  summarize(mean_delay = round(mean(dep_delay), 1)) %>%
  arrange(desc(mean_delay))
# July (month 7) and June (6) typically worst; December also bad

**Solution 4.4**

In [ ]:
flights_clean %>%
  group_by(origin, dest) %>%
  summarize(n = n(), .groups = "drop") %>%
  arrange(desc(n)) %>%
  head(5)

---
## Step 5 · Visualize with ggplot2

**Solution 5.1**

In [ ]:
ggplot(flights_clean, aes(x = dep_delay)) +
  geom_histogram(bins = 60, fill = "#4E79A7", color = "white", alpha = 0.8) +
  coord_cartesian(xlim = c(-30, 180)) +
  labs(title = "Distribution of Departure Delays",
       x = "Departure Delay (minutes)", y = "Count") +
  theme_minimal()

**Solution 5.2**

In [ ]:
ggplot(flights_clean, aes(x = carrier, y = dep_delay, fill = carrier)) +
  geom_boxplot(outlier.alpha = 0.1) +
  coord_flip(ylim = c(-30, 120)) +
  labs(title = "Departure Delay by Carrier",
       x = "Carrier", y = "Departure Delay (minutes)") +
  theme_minimal() +
  theme(legend.position = "none")

**Solution 5.3**

In [ ]:
monthly <- flights_clean %>%
  group_by(month) %>%
  summarize(mean_arr_delay = mean(arr_delay)) %>%
  mutate(flag = ifelse(mean_arr_delay > 10, "High", "Low"))

ggplot(monthly, aes(x = factor(month), y = mean_arr_delay, fill = flag)) +
  geom_col() +
  scale_fill_manual(values = c("High" = "#E15759", "Low" = "#4E79A7")) +
  labs(title = "Mean Arrival Delay by Month",
       x = "Month", y = "Mean Arrival Delay (min)", fill = "Delay Level") +
  theme_minimal()

**Solution 5.4**

In [ ]:
ggplot(flights_clean %>% sample_n(10000),   # sample for speed
       aes(x = distance, y = air_time, color = origin)) +
  geom_point(alpha = 0.3, size = 1) +
  geom_smooth(method = "lm", se = FALSE) +
  labs(title = "Distance vs Air Time by Origin Airport",
       x = "Distance (miles)", y = "Air Time (minutes)", color = "Origin") +
  theme_minimal()
# Strong positive linear relationship — further flights take longer (obvious but confirms data quality)

---
## Step 6 · Linear Regression

**Solution 6.1**

In [ ]:
model1 <- lm(arr_delay ~ dep_delay, data = flights_clean)
summary(model1)
# Slope ~1.02: for every 1 extra minute of departure delay,
# arrival delay increases by ~1 minute — flights rarely make up time

**Solution 6.2**

In [ ]:
model2 <- lm(arr_delay ~ dep_delay + distance + air_time, data = flights_clean)
summary(model2)
# dep_delay: highly significant (p < 2e-16)
# distance and air_time: also significant — longer flights tend to recover more delay
# Adjusted R-squared ~0.86 — model explains 86% of variance in arrival delay

**Solution 6.3**

In [ ]:
set.seed(42)
train_idx    <- sample(1:nrow(flights_clean), size = 0.8 * nrow(flights_clean))
train_data   <- flights_clean[train_idx, ]
test_data    <- flights_clean[-train_idx, ]

model2_train <- lm(arr_delay ~ dep_delay + distance + air_time, data = train_data)
predictions  <- predict(model2_train, newdata = test_data)

library(Metrics)
cat("RMSE:", round(rmse(test_data$arr_delay, predictions), 2), "minutes\n")
# Typical RMSE ~12-14 minutes

**Solution 6.4 (Bonus)**

In [ ]:
results <- data.frame(
  actual    = test_data$arr_delay,
  predicted = predictions
)

ggplot(results %>% sample_n(5000), aes(x = actual, y = predicted)) +
  geom_point(alpha = 0.2, color = "#4E79A7") +
  geom_abline(slope = 1, intercept = 0, color = "#E15759", linewidth = 1.2) +
  coord_cartesian(xlim = c(-60, 200), ylim = c(-60, 200)) +
  labs(title = "Predicted vs Actual Arrival Delay",
       subtitle = "Red line = perfect prediction",
       x = "Actual Delay (min)", y = "Predicted Delay (min)") +
  theme_minimal()
# Points far from the line = large prediction errors (extreme weather, cascading delays)

---
## Step 7 · AI Collaboration — Reference Answers

**Solution 7.1** — Top 10 destinations by mean arrival delay

In [ ]:
top_dest <- flights_clean %>%
  group_by(dest) %>%
  summarize(mean_arr_delay = mean(arr_delay), n = n()) %>%
  filter(n > 100) %>%
  arrange(desc(mean_arr_delay)) %>%
  head(10)

ggplot(top_dest, aes(x = reorder(dest, mean_arr_delay), y = mean_arr_delay)) +
  geom_col(fill = "#E15759") +
  coord_flip() +
  labs(title = "Top 10 Destinations by Mean Arrival Delay",
       x = "Destination", y = "Mean Arrival Delay (min)") +
  theme_minimal()

**Solution 7.2** — ANOVA + Tukey test across carriers

In [ ]:
aov_model <- aov(dep_delay ~ carrier, data = flights_clean)
summary(aov_model)
# p < 2e-16: yes, carriers differ significantly in mean departure delay

tukey <- TukeyHSD(aov_model)
# Shows which specific carrier pairs differ significantly
plot(tukey, las = 1, cex.axis = 0.6)

---
*Instructor solutions — LONI/LSU HPC Workshop — Summer 2026 | hpc@loni.org*